# 补丁：多 seed 重跑 + 两个 baseline + 统计检验

基于你的 `run_experiment` 改的，加了：
1. `seed` 参数，支持多个 random seed 重复跑
2. 两个 baseline（Full data / Drop column），同样支持多 seed
3. 统计检验部分（配对 t-test / Wilcoxon）

把这些 cell 接在你已经有的 `injection_data`、`final_features`、`train_df`、`test_df` 后面跑。


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from scipy.stats import ttest_rel, wilcoxon


In [ ]:
df = pd.read_parquet(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/final_df_(distance)_v2.parquet"
)

final_features = [
    "total_price",
    "total_freight_value",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "item_count",
    "max_payment_installments",
    "average_distance_km"
]

experiment_columns = [
    "order_id",
    *final_features,
    "is_delayed"
]

model_df = df[experiment_columns].dropna().copy()

print(model_df.shape)
print(model_df.isna().sum().sum())

(95960, 10)
0


In [11]:
print(model_df.shape)

(95960, 10)


In [10]:
model_df.isna().sum()


order_id                     0
total_price                  0
total_freight_value          0
total_product_weight_g       0
average_product_length_cm    0
average_product_height_cm    0
item_count                   0
max_payment_installments     0
average_distance_km          0
is_delayed                   0
dtype: int64

In [ ]:
cols = final_features + ["is_delayed"]
print(df.shape[0])
print(df[cols].isna().sum())

99418
total_price                   775
total_freight_value           775
total_product_weight_g        775
average_product_length_cm     791
average_product_height_cm     791
item_count                    775
max_payment_installments        1
average_distance_km          1264
is_delayed                   2965
dtype: int64


In [3]:
model_df

,order_id,total_price,total_freight_value,total_product_weight_g,average_product_length_cm,average_product_height_cm,item_count,max_payment_installments,average_distance_km,is_delayed
0,e481f51cbdc54678b7cc49136f2d6af7,29.99,8.72,500.0,19.0,8.0,1.0,3.0,18.576110,False
1,53cdb2fc8bc7dce0b6741e2150273451,118.70,22.76,400.0,19.0,13.0,1.0,1.0,851.495069,False
2,47770eb9100c2d0c44946d9cf07ec65d,159.90,19.22,420.0,24.0,19.0,1.0,3.0,514.410666,False
3,949d5b44dbf5de918fe9c16f97b45f8a,45.00,27.20,450.0,30.0,10.0,1.0,1.0,1822.226336,False
4,ad21c59c0840e6cb83a9ceb5573f8159,19.90,8.72,250.0,51.0,15.0,1.0,1.0,29.676625,False
...,...,...,...,...,...,...,...,...,...,...
99413,9c5dedf39a927c1b2549525ed64a053c,72.00,13.08,1175.0,22.0,13.0,1.0,3.0,69.755064,False
99414,63943bddc261676b46f01ca7ac2f7bd8,174.90,20.10,4950.0,40.0,10.0,1.0,3.0,474.120037,False
99415,83c1379a015df1e13d02aae0204711ab,205.99,65.02,13300.0,32.0,90.0,1.0,5.0,967.847297,False
99416,11c177c8e97725db2631073c19f07b62,359.98,81.18,13100.0,20.0,20.0,2.0,4.0,370.404482,False


In [4]:
print(model_df.describe())
print(model_df["is_delayed"].value_counts())
print(model_df["is_delayed"].value_counts(normalize=True))

        total_price  total_freight_value  total_product_weight_g  \
count  95960.000000         95960.000000            95960.000000   
mean     137.069822            22.777101             2384.566267   
std      209.234391            21.581735             4767.831835   
min        0.850000             0.000000                0.000000   
25%       45.900000            13.830000              300.000000   
50%       86.000000            17.160000              750.000000   
75%      149.900000            24.000000             2050.000000   
max    13440.000000          1794.960000           184400.000000   

       average_product_length_cm  average_product_height_cm    item_count  \
count               95960.000000               95960.000000  95960.000000   
mean                   30.077695                  16.462999      1.142070   
std                    16.036169                  13.250083      0.538865   
min                     7.000000                   2.000000      1.000000   
25

In [28]:
from pathlib import Path

data_folder = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/exp_data")

injection_data = {}

for file in data_folder.glob("*.parquet"):
    injection_data[file.stem] = pd.read_parquet(file)

print(injection_data.keys())

dict_keys(['mar_50', 'mnar_50', 'mcar_30', 'mar_30', 'mnar_30', 'mcar_50', 'mcar_10', 'mar_10', 'mnar_10'])


In [29]:
downstream_results = []

## 修好 + 加 seed 参数的 `run_experiment`

In [35]:
def run_experiment(injected_df, mechanism, missing_rate, method,
                    seed=42, train_order_ids=None, test_order_ids=None):
    injected_df = injected_df.copy()

    train_df_exp = injected_df[injected_df["order_id"].isin(train_order_ids)].copy()
    test_df_exp = injected_df[injected_df["order_id"].isin(test_order_ids)].copy()

    # =========================
    # Imputation
    # =========================
    if method == "Mean":
        value = train_df_exp["total_freight_value"].mean()
        train_df_exp["total_freight_value"] = train_df_exp["total_freight_value"].fillna(value)
        test_df_exp["total_freight_value"] = test_df_exp["total_freight_value"].fillna(value)

    elif method == "Median":
        value = train_df_exp["total_freight_value"].median()
        train_df_exp["total_freight_value"] = train_df_exp["total_freight_value"].fillna(value)
        test_df_exp["total_freight_value"] = test_df_exp["total_freight_value"].fillna(value)

    elif method in ["KNN", "MICE"]:
        scaler = StandardScaler()

        X_train = train_df_exp[final_features].copy()
        X_test = test_df_exp[final_features].copy()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        if method == "KNN":
            imputer = KNNImputer(n_neighbors=5, weights="uniform")
        else:
            imputer = IterativeImputer(max_iter=10, random_state=seed)

        X_train_imputed = imputer.fit_transform(X_train_scaled)
        X_test_imputed = imputer.transform(X_test_scaled)

        X_train_imputed = scaler.inverse_transform(X_train_imputed)
        X_test_imputed = scaler.inverse_transform(X_test_imputed)

        train_df_exp[final_features] = X_train_imputed
        test_df_exp[final_features] = X_test_imputed

    elif method not in ["None",'Full_data'] :
        raise ValueError(f"Unknown imputation method: {method}")
    # method == "None" -> 跳过填补，用于 baseline

    # =========================
    # Prepare X / y
    # =========================
    features_used = final_features
    X_train = train_df_exp[features_used]
    y_train = train_df_exp["is_delayed"]
    X_test = test_df_exp[features_used]
    y_test = test_df_exp["is_delayed"]

    # =========================
    # Class imbalance + model
    # =========================
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    xgb_model = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=seed, eval_metric="logloss", n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )
    xgb_model.fit(X_train, y_train)

    y_pred = xgb_model.predict(X_test)
    y_prob = xgb_model.predict_proba(X_test)[:, 1]

    return {
        "seed": seed,
        "mechanism": mechanism,
        "missing_rate": missing_rate,
        "method": method,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }


## Standalone helper function for Drop_column (removes the entire feature column, unlike the imputation logic in run_experiment)

In [36]:
def run_drop_column_baseline(mechanism, missing_rate, seed, train_order_ids, test_order_ids):
    features_no_target = [f for f in final_features if f != "total_freight_value"]

    train_df_dc = model_df[model_df["order_id"].isin(train_order_ids)].copy()
    test_df_dc = model_df[model_df["order_id"].isin(test_order_ids)].copy()

    X_train = train_df_dc[features_no_target]
    y_train = train_df_dc["is_delayed"]
    X_test = test_df_dc[features_no_target]
    y_test = test_df_dc["is_delayed"]

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    xgb_model = XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=seed, eval_metric="logloss", n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )
    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    y_prob = xgb_model.predict_proba(X_test)[:, 1]

    return {
        "seed": seed,
        "mechanism": mechanism,
        "missing_rate": missing_rate,
        "method": "Drop_column",
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }


## 多 seed 主循环：36 组 + 2 个 baseline，各跑 5 个 seed = 190 行

**每个 seed 都重新切一次 train/test split**（不是固定用同一份 train_idx/test_idx），
这样才是真正独立的重复实验，而不是同一份数据换个模型 random_state 而已。


In [37]:
SEEDS = [42, 1, 7, 21, 100]

downstream_results = []

for seed in SEEDS:
    train_df_s, test_df_s = train_test_split(
        model_df,
        test_size=0.2,
        random_state=seed,
        stratify=model_df["is_delayed"]
    )
    train_order_ids = set(train_df_s["order_id"])
    test_order_ids = set(test_df_s["order_id"])

    # 36 组：9 个注入数据集 × 4 种填补方法
    for name, injected_df in injection_data.items():
        mechanism, rate_str = name.split("_")
        mechanism = mechanism.upper()
        rate = int(rate_str) / 100

        for method in ["Mean", "Median", "KNN", "MICE"]:
            result = run_experiment(
                injected_df, mechanism, rate, method,
                seed=seed,
                train_order_ids=train_order_ids,
                test_order_ids=test_order_ids
            )
            downstream_results.append(result)

    # Baseline 1: Full data
    result = run_experiment(
        model_df, mechanism="Baseline", missing_rate=0.0, method="Full_data",
        seed=seed,
        train_order_ids=train_order_ids,
        test_order_ids=test_order_ids
    )
    downstream_results.append(result)

    # Baseline 2: Drop column
    result = run_drop_column_baseline(
        mechanism="Baseline", missing_rate=0.0, seed=seed,
        train_order_ids=train_order_ids, test_order_ids=test_order_ids
    )
    downstream_results.append(result)

    print(f"Seed {seed} done, total rows so far: {len(downstream_results)}")

downstream_results_df = pd.DataFrame(downstream_results)
print("Total rows:", len(downstream_results_df))
downstream_results_df.head()


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Seed 42 done, total rows so far: 38


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Seed 1 done, total rows so far: 76


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

...

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Seed 21 done, total rows so far: 152


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Seed 100 done, total rows so far: 190
Total rows: 190


,seed,mechanism,missing_rate,method,accuracy,precision,recall,f1,roc_auc
0,42,MAR,0.5,Mean,0.648708,0.108173,0.460450,0.175190,0.589667
1,42,MAR,0.5,Median,0.648708,0.108646,0.463023,0.175996,0.592996
2,42,MAR,0.5,KNN,0.651365,0.111380,0.473312,0.180326,0.587873
3,42,MAR,0.5,MICE,0.656003,0.110511,0.460450,0.178242,0.587959
4,42,MNAR,0.5,Mean,0.649229,0.107625,0.456592,0.174190,0.590209


## 保存原始（每个 seed 一行）结果

In [38]:
downstream_results_df.to_csv(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results_all_seeds.csv",
    index=False
)


## 汇总：每个 mechanism/rate/method 组合的均值 ± 标准差

这是论文里应该报告的表格形式，而不是单一数字。


In [39]:
summary_df = (
    downstream_results_df
    .groupby(["mechanism", "missing_rate", "method"])
    .agg(
        accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"), precision_std=("precision", "std"),
        recall_mean=("recall", "mean"), recall_std=("recall", "std"),
        f1_mean=("f1", "mean"), f1_std=("f1", "std"),
        roc_auc_mean=("roc_auc", "mean"), roc_auc_std=("roc_auc", "std"),
    )
    .reset_index()
)

summary_df.to_csv(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results_summary.csv",
    index=False
)

summary_df


,mechanism,missing_rate,method,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Baseline,0.0,Drop_column,0.650459,0.009861,0.108201,0.003144,0.457363,0.012390,0.174980,0.004559,0.584970,0.005395
1,Baseline,0.0,Full_data,0.639225,0.006161,0.108520,0.002825,0.478585,0.017084,0.176913,0.004743,0.592370,0.006143
2,MAR,0.1,KNN,0.639840,0.007178,0.108535,0.003022,0.477685,0.020736,0.176865,0.005226,0.590689,0.005369
3,MAR,0.1,MICE,0.639569,0.005124,0.108652,0.002364,0.478714,0.013631,0.177100,0.003912,0.590884,0.003882
4,MAR,0.1,Mean,0.636150,0.006326,0.108337,0.001591,0.482830,0.015153,0.176955,0.002921,0.592478,0.005586
5,MAR,0.1,Median,0.634754,0.008143,0.107950,0.003135,0.482830,0.015153,0.176437,0.004895,0.591585,0.005347
6,MAR,0.3,KNN,0.640694,0.008111,0.107712,0.002998,0.471511,0.018389,0.175347,0.004939,0.589425,0.004645
7,MAR,0.3,MICE,0.640652,0.005151,0.108051,0.002957,0.473441,0.013284,0.175941,0.004722,0.589534,0.004709
8,MAR,0.3,Mean,0.641861,0.008762,0.108752,0.002640,0.475113,0.008892,0.176976,0.003707,0.591769,0.004854
9,MAR,0.3,Median,0.641986,0.006358,0.108126,0.002667,0.471768,0.020125,0.175916,0.004772,0.590007,0.005521


## 统计检验：配对比较

思路：同一个 seed 下，不同方法用的是同一份 train/test split，所以是配对数据，
用配对 t-test（假设近似正态）或 Wilcoxon signed-rank test（更稳健，不假设正态，
5 个 seed 样本量很小时更推荐这个）比较两两方法之间 AUC 有没有显著差异。

下面示范几个最关键的对比，不需要 36 个组合两两都测：
1. 各填补方法之间（在同一个 mechanism/rate 下）
2. 某个填补方法 vs Baseline_Full_data（填补能不能接近"完美"的情况）
3. 某个填补方法 vs Baseline_Drop_column（填补是不是至少比完全不用这个特征好）


In [40]:
def paired_compare(df, mechanism, missing_rate, method_a, method_b, metric="roc_auc"):
    a = (
        df[(df["mechanism"] == mechanism) &
           (df["missing_rate"] == missing_rate) &
           (df["method"] == method_a)]
        .sort_values("seed")[metric]
        .to_numpy()
    )
    b = (
        df[(df["mechanism"] == mechanism) &
           (df["missing_rate"] == missing_rate) &
           (df["method"] == method_b)]
        .sort_values("seed")[metric]
        .to_numpy()
    )

    assert len(a) == len(b), f"Sample size mismatch: {len(a)} vs {len(b)}"

    t_stat, t_p = ttest_rel(a, b)
    try:
        w_stat, w_p = wilcoxon(a, b)
    except ValueError:
        # 全部差值为 0 时 wilcoxon 会报错
        w_stat, w_p = np.nan, np.nan

    print(f"{mechanism} / {missing_rate:.0%} : {method_a} vs {method_b} ({metric})")
    print(f"  {method_a} mean={a.mean():.4f}  {method_b} mean={b.mean():.4f}  diff={a.mean()-b.mean():.4f}")
    print(f"  paired t-test   p = {t_p:.4f}")
    print(f"  wilcoxon test   p = {w_p:.4f}")
    print()

    return {"t_p": t_p, "wilcoxon_p": w_p, "mean_a": a.mean(), "mean_b": b.mean()}


In [41]:
# 示例 1：MICE vs KNN，在 MAR/10% 下
paired_compare(downstream_results_df, "MAR", 0.10, "MICE", "KNN")

# 示例 2：MICE vs Mean，在 MAR/10% 下
paired_compare(downstream_results_df, "MAR", 0.10, "MICE", "Mean")

# 示例 3：MICE (MAR/10%) vs Baseline Full data
mice_df = downstream_results_df[
    (downstream_results_df["mechanism"] == "MAR") &
    (downstream_results_df["missing_rate"] == 0.10) &
    (downstream_results_df["method"] == "MICE")
][["seed", "roc_auc"]].rename(columns={"roc_auc": "mice_auc"})

baseline_df = downstream_results_df[
    (downstream_results_df["mechanism"] == "Baseline") &
    (downstream_results_df["method"] == "None")
][["seed", "roc_auc"]].rename(columns={"roc_auc": "baseline_auc"})

merged = mice_df.merge(baseline_df, on="seed", how="inner")

t_stat, t_p = ttest_rel(merged["mice_auc"], merged["baseline_auc"])
print("MICE(MAR/10%) vs Baseline_Full_data (roc_auc)")
print(f"  MICE mean={merged['mice_auc'].mean():.4f}  Baseline mean={merged['baseline_auc'].mean():.4f}")
print(f"  paired t-test p = {t_p:.4f}")


MAR / 10% : MICE vs KNN (roc_auc)
  MICE mean=0.5909  KNN mean=0.5907  diff=0.0002
  paired t-test   p = 0.9047
  wilcoxon test   p = 0.6250

MAR / 10% : MICE vs Mean (roc_auc)
  MICE mean=0.5909  Mean mean=0.5925  diff=-0.0016
  paired t-test   p = 0.1149
  wilcoxon test   p = 0.1875

MICE(MAR/10%) vs Baseline_Full_data (roc_auc)
  MICE mean=nan  Baseline mean=nan
  paired t-test p = nan


/var/folders/b0/srn_nzdx2p10qcyxh4cgc92r0000gn/T/ipykernel_22465/2898390046.py:21: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  t_stat, t_p = ttest_rel(merged["mice_auc"], merged["baseline_auc"])


### 怎么解读这几行 p-value

- **p < 0.05**：两者之间的差异在统计上显著，可以说"确实不一样"
- **p >= 0.05**：没有足够证据说两者不一样——差异可能只是随机波动

根据前面单次跑的结果（AUC 全部挤在 0.587-0.593 之间），预期大部分方法之间的
p-value 会 >= 0.05（差异不显著），这跟"填补方法对下游任务影响有限"这个结论是一致的。
如果真的测出来大部分都不显著，这不是坏事——是一个站得住脚、可以直接写进论文的结论，
用 5 个 seed 的检验结果撑住它，比只凭单次跑的数字肉眼比较更有说服力。
